# N8 — Execute and Monitor

## Decision question

Who owns each action, what evidence proves completion, and which measurable
trigger forces the team to escalate or change course?


In [ ]:
from pathlib import Path
import json
import sys

# Find the public package locally. A fresh Colab runtime downloads the same
# participant-safe assets from the repository.
for candidate in [Path.cwd(), *Path.cwd().parents]:
    for source_candidate in (candidate / 'src', candidate / 'CFOPackV002' / 'src'):
        if (source_candidate / 'workshop_bootstrap.py').exists():
            sys.path.insert(0, str(source_candidate))
            break

try:
    from workshop_bootstrap import bootstrap
except ImportError:
    from urllib.request import urlopen
    bootstrap_url = (
        'https://raw.githubusercontent.com/VinayaSharada/'
        'KateelLearningDemosToStudents/cfopack-v002-v2.1.0-beta.1/CFOPackV002/src/workshop_bootstrap.py'
    )
    namespace = {}
    exec(compile(urlopen(bootstrap_url).read(), bootstrap_url, 'exec'), namespace)
    bootstrap = namespace['bootstrap']

ROOT, OUTPUT_DIR = bootstrap()
from cfopack_v002 import (
    analyze_fx,
    default_decisions,
    load_inputs,
    load_manifest,
    reveal_team_shock,
    run_pipeline,
)
import workshop_visuals as viz
import pandas as pd
try:
    from IPython.display import Markdown, display
except ImportError:
    # Keep the notebooks runnable from a minimal local Python environment as
    # well as Colab/Jupyter. Rich notebook rendering remains the default.
    def Markdown(value):
        return value

    def display(value):
        print(value)

manifest = load_manifest(ROOT / 'config' / 'scenario_manifest.json')
decision_file = OUTPUT_DIR / 'N0_team_decisions.json'
if decision_file.exists():
    DECISIONS = json.loads(decision_file.read_text(encoding='utf-8'))
else:
    DECISIONS = default_decisions(manifest)


In [ ]:
data = load_inputs(ROOT / 'data' / 'synthetic')
viz.data_snapshot(data, OUTPUT_DIR, 'N8')


In [ ]:
approval_file = OUTPUT_DIR / 'N7_cfo_approval.json'
if approval_file.exists():
    approval = json.loads(approval_file.read_text(encoding='utf-8'))
else:
    approval = {'status': 'approve', 'note': 'Standalone N8 recovery default'}
if approval['status'] != 'approve':
    raise RuntimeError('Execution is blocked until the CFO decision is approved in N7.')


In [ ]:
summary = run_pipeline(ROOT, OUTPUT_DIR, DECISIONS)
print(f"Scenario {summary['scenario_version']} calculated for {DECISIONS['team_name']} (model cache: {'hit' if summary['model_cache_hit'] else 'rebuilt'})")


## 30-day action plan and scorecard


In [ ]:
action_plan = pd.read_csv(OUTPUT_DIR / 'N8_action_plan.csv')
scorecard = pd.read_csv(OUTPUT_DIR / 'N8_monitoring_scorecard.csv')
display(action_plan)
display(scorecard)
viz.execution_chart(action_plan, OUTPUT_DIR)


## Final team commitment

Before the CFO defence, confirm:

- Every action has one accountable owner.
- Every action has observable completion evidence.
- Every trigger has a named escalation path and response time.
- Daily actual receipts and outflows will replace forecast values.
- The team has named the first decision it will revisit if the scenario worsens.

Your workshop outcome is the defended decision and executable control system—not
the fact that every notebook ran successfully.


In [ ]:
DECISIONS['collections_receipt_floor'] = 0.90
decision_file.write_text(json.dumps(DECISIONS, indent=2), encoding='utf-8')
summary = run_pipeline(ROOT, OUTPUT_DIR, DECISIONS)
display(pd.read_csv(OUTPUT_DIR / 'decision_ledger.csv'))
print('Execution controls recorded and linked to the approved decision.')


### Before moving on

Record your interpretation in the participant workbook. Do not copy a chart
without also recording the assumption and decision it supports.
